In [1]:
# import libraries for reading data
import pandas as pd
import matplotlib.pyplot as plt
import os
from PIL import Image
import cv2
import re
import torch
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from torchvision import transforms, models
import ast
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights, efficientnet_v2_s, EfficientNet_V2_S_Weights
import torch.nn as nn
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from concurrent.futures import ProcessPoolExecutor
from PIL import ImageEnhance, Image
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler
from concurrent.futures import ThreadPoolExecutor
import torch.multiprocessing as mp
from torchvision.models import convnext_large, ConvNeXt_Large_Weights
from sklearn.model_selection import GroupKFold
import gc
mp.set_sharing_strategy('file_system')

<jemalloc>: Unsupported system page size


### Einlesen der Daten und Übersicht über die Daten

In [2]:
# data paths
train1_images_path = "/datasets/multi-view-pig-posture-recognition/train1_images"
train2_images_path = "/datasets/multi-view-pig-posture-recognition/train2_images"
test_images_path = "/datasets/multi-view-pig-posture-recognition/test_images"

# csv path with row_id, image_id, width, height, bbox, class_id
train1_csv_path = "/datasets/multi-view-pig-posture-recognition/train1.csv"
train2_csv_path = "/datasets/multi-view-pig-posture-recognition/train2.csv"
test_csv_path = "/datasets/multi-view-pig-posture-recognition/test.csv"

# txt file path
pig_posture_txt = "/datasets/multi-view-pig-posture-recognition/pig_posture_classes.txt"

In [3]:
# read all files and show statistics and content of csv files, column names etc.
# read csv files
train1_df = pd.read_csv(train1_csv_path)
train2_df = pd.read_csv(train2_csv_path)
test_df = pd.read_csv(test_csv_path)

# show column names of csv files
print("\nTrain1 CSV Columns:")
print(train1_df.columns)
print("\nTrain2 CSV Columns:")
print(train2_df.columns)
print("\nTest CSV Columns:")
print(test_df.columns)

# show content of txt file
with open(pig_posture_txt, 'r') as f:
    pig_posture_content = f.read()

# show numbers of unique image_ids, row_ids in train1, train2 and test csv files
print("\nNumber of unique image_ids in Train1 CSV:", train1_df['image_id'].nunique())
print("Number of unique image_ids in Train2 CSV:", train2_df['image_id'].nunique())
print("Number of unique row_ids in Train1 CSV:", train1_df['row_id'].nunique())
print("Number of unique row_ids in Train2 CSV:", train2_df['row_id'].nunique())
print("\nPig Posture Classes:")
print(pig_posture_content)




Train1 CSV Columns:
Index(['row_id', 'image_id', 'width', 'height', 'bbox', 'class_id'], dtype='object')

Train2 CSV Columns:
Index(['row_id', 'image_id', 'width', 'height', 'bbox', 'class_id'], dtype='object')

Test CSV Columns:
Index(['row_id', 'image_id', 'width', 'height', 'bbox'], dtype='object')

Number of unique image_ids in Train1 CSV: 3090
Number of unique image_ids in Train2 CSV: 3150
Number of unique row_ids in Train1 CSV: 22934
Number of unique row_ids in Train2 CSV: 23450

Pig Posture Classes:
Lateral_lying_left
Lateral_lying_right
Sitting
Standing
Sternal_lying



In [4]:
train1_df.head()

,row_id,image_id,width,height,bbox,class_id
0,train_pen1_orb_cam1_20250108_085204_0000,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[1031.5,368.0,349.0,435.0]",0
1,train_pen1_orb_cam1_20250108_085204_0001,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[1278.5,428.0,233.0,438.0]",4
2,train_pen1_orb_cam1_20250108_085204_0002,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[732.0,137.5,342.0,198.0]",1
3,train_pen1_orb_cam1_20250108_085204_0003,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[830.0,169.0,370.0,263.0]",0
4,train_pen1_orb_cam1_20250108_085204_0004,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[611.5,314.8,381.5,386.6]",3


In [5]:
train2_df.head()

,row_id,image_id,width,height,bbox,class_id
0,train_pen1_orb_cam1_20250108_085204_0000,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[1031.5,368.0,349.0,435.0]",0
1,train_pen1_orb_cam1_20250108_085204_0001,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[1278.5,428.0,233.0,438.0]",4
2,train_pen1_orb_cam1_20250108_085204_0002,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[732.0,137.5,342.0,198.0]",1
3,train_pen1_orb_cam1_20250108_085204_0003,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[830.0,169.0,370.0,263.0]",0
4,train_pen1_orb_cam1_20250108_085204_0004,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[611.5,314.8,381.5,386.6]",3


### EDA 
Class Definitions:

0 — Lateral_lying_left

1 — Lateral_lying_right

2 — Sitting

3 — Standing

4 — Sternal_lying

### Klassen sind stark unausgewogen, insbesondere Sitting Class id = 2. Gegenmaßnahme ist notwendig, um die Minderheitsklasse nicht zu vernachlässigen.

In [6]:
# check if there are any missing values in train1 and train2 csv files
print("\nMissing values in Train1 CSV:")
print(train1_df.isnull().sum())
print("\nMissing values in Train2 CSV:")
print(train2_df.isnull().sum())


Missing values in Train1 CSV:
row_id      0
image_id    0
width       0
height      0
bbox        0
class_id    0
dtype: int64

Missing values in Train2 CSV:
row_id      0
image_id    0
width       0
height      0
bbox        0
class_id    0
dtype: int64


In [7]:
# check if unique values in "height" and "weight" columns in train1 and train2 csv files are the same
print("\nUnique values in 'height' column in Train1 CSV:")
print(train1_df['height'].unique())
print("\nUnique values in 'height' column in Train2 CSV:")
print(train2_df['height'].unique())
print("\nUnique values in 'width' column in Train1 CSV:")
print(train1_df['width'].unique())
print("\nUnique values in 'width' column in Train2 CSV:")
print(train2_df['width'].unique())


Unique values in 'height' column in Train1 CSV:
[1080  720 1520]

Unique values in 'height' column in Train2 CSV:
[1080  720 1520]

Unique values in 'width' column in Train1 CSV:
[1920 1280 2688]

Unique values in 'width' column in Train2 CSV:
[1920 1280 2688]


### Die Bilder liegen nur in drei Auflösungen vor: 1280 x 720, 1920 x 1080, 2688 x 1520. Vorverarbeitung ist konsistent planbar. 

### Fazit: Die EDA zeigt, dass die Klassen in den Trainingsdaten relativ ausgewogen verteilt sind, was für das Training eines Modells vorteilhaft ist. Es gibt keine fehlenden Werte in den CSV-Dateien, und die Bildgrößen sind konsistent. Die Analyse der Bildqualität anhand von Blur-Score und Helligkeit zeigt eine gewisse Variation. Im nächsten Schritt möchte ich die Kameras trennen und die Bilder entsprechend der Kamera analysieren, um mögliche Unterschiede in der Bildqualität oder den Aufnahmewinkeln zu identifizieren.

### Trennung der Kameras anhand der Bildnamen nur in Train1

In [8]:
def extract_pen_id(s: str):
    m = re.search(r'^(pen\d+)', s)
    return m.group(1) if m else None

def extract_camera_type(s: str):
    m = re.search(r'_(orb|tur)_', s)
    return m.group(1) if m else None

def extract_camera_number(s: str):
    m = re.search(r'cam(\d+)', s)
    return m.group(1) if m else None

# df1 = train1.csv als DataFrame; ersetze 'FILENAME_COL' durch deine Spalte (z. B. 'image', 'file_name', ...).
FILENAME_COL = "image_id"
train1_df["pen_id"]        = train1_df[FILENAME_COL].apply(extract_pen_id)
train1_df["camera_type"]   = train1_df[FILENAME_COL].apply(extract_camera_type)
train1_df["camera_number"] = train1_df[FILENAME_COL].apply(extract_camera_number)
train1_df["camera_view_id"]      = train1_df["pen_id"] + "_" + train1_df["camera_type"] + "_cam" + train1_df["camera_number"]


In [9]:
train1_df_distribution = train1_df['class_id'].value_counts().sort_index()
train2_df_distribution = train2_df['class_id'].value_counts().sort_index()

print("\nClass Distribution in Train1 CSV:")
print(train1_df_distribution)
print("\nClass Distribution in Train2 CSV:")
print(train2_df_distribution)


Class Distribution in Train1 CSV:
class_id
0    3053
1    3376
2     680
3    9617
4    6208
Name: count, dtype: int64

Class Distribution in Train2 CSV:
class_id
0    3083
1    3435
2     695
3    9928
4    6309
Name: count, dtype: int64


### Trennung der Kameras anhand der Bildnamen nur in Train2

In [10]:
train2_df["pen_id"]        = train2_df["image_id"].apply(extract_pen_id)
train2_df["camera_type"]   = train2_df["image_id"].apply(extract_camera_type)
train2_df["camera_number"] = train2_df["image_id"].apply(extract_camera_number)
train2_df["camera_view_id"]      = train2_df["pen_id"] + "_" + train2_df["camera_type"] + "_cam" + train2_df["camera_number"]


### Erster Versuch eines Modelltrainings mit dem Modell "MobileNetv3". Vorbereitungen treffen mit transforms. 

In [11]:
# transforms for data augmentation and data preprocessing
img_size = (224, 224)
normalize_mean = [0.485, 0.456, 0.406]
normalize_std = [0.229, 0.224, 0.225]


train_transforms = transforms.Compose([
    transforms.Resize(img_size),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=normalize_mean, std=normalize_std),
])

val_transforms = transforms.Compose([
    transforms.Resize(img_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=normalize_mean, std=normalize_std),
])



In [12]:
class PigCropDataset(Dataset):
    def __init__(self, df, image_dir, transform=None, preload=True):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.images = []
        self.labels = []
        
        if preload:
            print(f"Lade {len(self.df)} Bilder mit 120 Cores in den RAM...")
            
            def load_single_image(idx):
                r = self.df.iloc[idx]
                p = os.path.join(self.image_dir, r["image_id"])
                # Direktes Laden und Zuschneiden
                img = Image.open(p).convert("RGB")
                bbox = ast.literal_eval(r["bbox"]) if isinstance(r["bbox"], str) else r["bbox"]
                x, y, w, h = bbox
                # Resize hier spart massiv RAM und CPU-Zeit beim Training
                crop = img.crop((x, y, x + w, y + h)).resize((224, 224))
                return crop, int(r["class_id"])

            # Wir nutzen 120 der 160 Cores, um das System nicht komplett zu blockieren
            with ThreadPoolExecutor(max_workers=120) as executor:
                results = list(tqdm(executor.map(load_single_image, range(len(self.df))), total=len(self.df)))
            
            self.images, self.labels = zip(*results)
            print("Preloading abgeschlossen. Daten liegen nun im RAM.")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Kein Disk-Zugriff mehr! Nur noch RAM-Zugriff.
        img = self.images[idx]
        label = self.labels[idx]
        
        if self.transform:
            img = self.transform(img)
            
        return img, torch.tensor(label, dtype=torch.long)


In [13]:
sampleDS = PigCropDataset(train2_df, train2_images_path, transform=train_transforms)
loader = DataLoader(sampleDS, batch_size=9, shuffle=True)
for x, y in loader:
    print("Batch Image Shape:", x.shape)  # erwartet: [9, 3, 224, 224]
    print("Batch Label Shape:", y.shape)  # erwartet: [9]
    print("First Label:", y[0].item())    # 0..4
    break


Lade 23450 Bilder mit 120 Cores in den RAM...


100%|██████████| 23450/23450 [00:13<00:00, 1711.40it/s]  


Preloading abgeschlossen. Daten liegen nun im RAM.
Batch Image Shape: torch.Size([9, 3, 224, 224])
Batch Label Shape: torch.Size([9])
First Label: 3


In [15]:
class PigPostureConvNeXt(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        weights = ConvNeXt_Large_Weights.DEFAULT
        self.backbone = convnext_large(weights=weights).features
        
        in_features = 1536 # ConvNeXt-Large Standard
        
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.LayerNorm(in_features), 
            nn.Linear(in_features, 512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.backbone(x))

# Instanziierung (wie gehabt)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PigPostureConvNeXt(num_classes=5)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model = model.to(device)

### Einfügen einer Metrik, um mit der Klassenverteilung besser ausheben zu können. Den unterrepräsentierten Klassen eine höhere Gewichtung geben, um die Ungleichheit der Klassenverteilung zu adressieren. 

In [16]:
class Learner:
    def __init__(self, model, train_dl, val_dl, device=None):
        self.model = model
        self.train_dl = train_dl
        self.val_dl = val_dl
        self.device = device
        
        self.model = self.model.to(self.device)
        self.loss_fn_classifier = nn.CrossEntropyLoss()
        self.best_acc = 0
        self.scaler = torch.cuda.amp.GradScaler() # NEU: Skalierung für Mixed Precision
        self.freeze()
        
    def freeze(self):
        actual_model = self.model.module if isinstance(self.model, nn.DataParallel) else self.model
        for param in actual_model.backbone.parameters():
            param.requires_grad = False
        
    def unfreeze(self):
        actual_model = self.model.module if isinstance(self.model, nn.DataParallel) else self.model
        for param in actual_model.backbone.parameters():
            param.requires_grad = True

    def fit(self, epochs, lr=1e-3):
        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=lr)
        self.scheduler = torch.optim.lr_scheduler.OneCycleLR(
            self.optimizer, max_lr=lr*10, total_steps=epochs*len(self.train_dl)
        )
        
        for epoch in range(epochs):
            self.model.train()
            # Fortschrittsbalken für das Training
            pbar = tqdm(self.train_dl, desc=f"Epoch {epoch+1}/{epochs}", leave=False)
            train_loss = 0
            
            for xb, yb in pbar:
                xb, yb = xb.to(self.device, non_blocking=True), yb.to(self.device, non_blocking=True)
                self.optimizer.zero_grad()
                with torch.cuda.amp.autocast():
                    preds = self.model(xb)
                    loss = self.loss_fn_classifier(preds, yb)
                
                self.scaler.scale(loss).backward()
                self.scaler.step(self.optimizer)
                self.scaler.update()
                self.scheduler.step()
                train_loss += loss.item()

            # --- VALIDIERUNG nach der Epoche ---
            self.model.eval()
            val_correct = 0
            with torch.no_grad():
                with torch.cuda.amp.autocast():
                    for xb, yb in self.val_dl:
                        xb, yb = xb.to(self.device, non_blocking=True), yb.to(self.device, non_blocking=True)
                        preds = self.model(xb)
                        val_correct += (preds.argmax(1) == yb).sum().item()
            
            acc = val_correct / len(self.val_dl.dataset)
            if acc > self.best_acc: self.best_acc = acc
            
            # SICHERE AUSGABE DER ACCURACY
            print(f"Epoche {epoch+1}: Train-Loss: {train_loss/len(self.train_dl):.4f} | Val-Accuracy: {acc:.4f} | Beste: {self.best_acc:.4f}")

In [17]:
# Wir nutzen 5 Folds, trainieren hier aber erst einmal Fold 1
gkf = GroupKFold(n_splits=5)
groups = train2_df['camera_view_id']

# Hol dir die Indizes für den ersten Fold
train_idx, val_idx = next(gkf.split(train2_df, groups=groups))

train_data = train2_df.iloc[train_idx].copy()
val_data = train2_df.iloc[val_idx].copy()

# Oversampling für Klasse 2 beibehalten
sitting_samples = train_data[train_data['class_id'] == 2]
train_data = pd.concat([train_data] + [sitting_samples] * 4, ignore_index=True)
train_data = train_data.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
## --- PHASE 1: Classifier Training ---
# Batch Size 256 (64 pro Karte) für den Anfang sicher
batch_size_p1 = 256 

train_ds = PigCropDataset(train_data, train2_images_path, transform=train_transforms)
val_ds = PigCropDataset(val_data, train2_images_path, transform=val_transforms)

train_dl = DataLoader(train_ds, batch_size=batch_size_p1, shuffle=True, num_workers=16, pin_memory=True)
val_dl = DataLoader(val_ds, batch_size=batch_size_p1, shuffle=False, num_workers=8, pin_memory=True)

learner = Learner(model, train_dl, val_dl, device=device)
print("Starte Phase 1 (ConvNeXt-Large Frozen)...")
learner.freeze()
learner.fit(epochs=15, lr=1e-3)

Lade 14565 Bilder mit 120 Cores in den RAM...


100%|██████████| 14565/14565 [00:02<00:00, 5697.91it/s] 


Preloading abgeschlossen. Daten liegen nun im RAM.
Lade 9713 Bilder mit 120 Cores in den RAM...


100%|██████████| 9713/9713 [00:10<00:00, 925.02it/s]  


Preloading abgeschlossen. Daten liegen nun im RAM.
Starte Phase 1 (ConvNeXt-Large Frozen)...


Epoche 1: Train-Loss: 0.9533 | Val-Accuracy: 0.5868 | Beste: 0.5868


Epoche 2: Train-Loss: 0.6080 | Val-Accuracy: 0.6108 | Beste: 0.6108


Epoche 3: Train-Loss: 0.5769 | Val-Accuracy: 0.6198 | Beste: 0.6198


Epoche 4: Train-Loss: 0.5984 | Val-Accuracy: 0.5120 | Beste: 0.6198


In [ ]:
# --- PHASE 2: Full Model Fine-Tuning ---
# WICHTIG: Batch Size weiter senken für Unfreeze (Gradienten brauchen Platz!)
batch_size_p2 = 128 # 32 pro Karte

torch.cuda.empty_cache()
gc.collect()

learner.train_dl = DataLoader(train_ds, batch_size=batch_size_p2, shuffle=True, num_workers=16, pin_memory=True)
learner.val_dl = DataLoader(val_ds, batch_size=batch_size_p2, shuffle=False, num_workers=8, pin_memory=True)

print("Starte Phase 2 (Full Model Unfrozen - 35 Epochen)...")
learner.unfreeze()
learner.fit(epochs=80, lr=5e-5)

In [ ]:
# 1. Test-Daten vorbereiten
test_df_copy = test_df.copy()
test_df_copy['class_id'] = 0 

# Sicherheitshalber eine moderate Batch-Size wählen (z.B. 512 oder 256)
batch_size_inference = 512 

test_ds = PigCropDataset(test_df_copy, test_images_path, transform=val_transforms)
test_dl = DataLoader(test_ds, batch_size=batch_size_inference, shuffle=False, 
                    num_workers=8, pin_memory=True)

# 2. Modell in den Vorhersage-Modus schalten
model.eval()
all_predictions = []

print(f"Erstelle Vorhersagen auf {device}...")

with torch.no_grad():
    # NUTZE AMP: Das spart VRAM und beschleunigt die Vorhersage auf V100 massiv
    with torch.cuda.amp.autocast():
        for xb, _ in tqdm(test_dl):
            # non_blocking=True für schnelleren Datentransfer
            xb = xb.to(device, non_blocking=True) 
            
            outputs = model(xb)
            
            # Klasse mit dem höchsten Wert
            _, preds = torch.max(outputs, 1)
            
            all_predictions.extend(preds.cpu().numpy())

# 3. Die finale CSV-Datei erstellen
submission = pd.DataFrame({
    'row_id': test_df['row_id'],
    'class_id': all_predictions
})

# Speichern
submission_name = '4th_submission.csv'
submission.to_csv(submission_name, index=False)

print(f"Erfolgreich! Die Datei '{submission_name}' mit {len(submission)} Zeilen wurde erstellt.")

### Erste Submission bei 10 Epochen hat eine Accuracy von 0.310 bei Kaggle erreicht. Erheblich von dem entfernt, was hier als Validation Accuracy von 0.83 zuletzt angezeigt wird

### Optimisierungsmaßnahmen, um einen besseren Score zu erreichen. Problematik der Verteilung der Klassen, unscharfe Bilder und Helligkeit erhöhen.